# Dropout — простой способ не переобучаться

**Статья:** Srivastava, Hinton, Krizhevsky, Sutskever, Salakhutdinov — *Dropout: A Simple Way to Prevent Neural Networks from Overfitting* (JMLR 2014).

🌳 **Узел дерева:** Deep Learning → регуляризация → dropout.

**Что строим и обучаем:** двухслойный ReLU-MLP на реальном датасете (sklearn digits, 8×8 рукописные цифры), c dropout, написанным с нуля. Реально обучаем end-to-end на CPU — обе руки ablation-а за десятки секунд.

**Компьют:** CPU, меньше минуты на обе руки.

**Вклад статьи, который ты реализуешь сама:** оператор `dropout_forward` в inverted-dropout конвенции — именно та несущая деталь, которую ablation включает/выключает.

**Parts:** (1) `dropout_forward` — механизм · (2) `train` — training loop · (3) ablation on/off и headline-контраст: узкий gap + выше test-acc.

## ⛏ Эксперименты-первыми (заполни ДО чтения метода)

Статья большая, но нам нужен один эффект: dropout уменьшает переобучение. Table 2 / Fig. 4: меньше train-test gap и ниже test error против той же сети без dropout (MNIST: ~1.6% → ~1.3% test error у 2-слойной сети).

> **(a) Что результаты реально показали vs abstract?** Abstract обещает «аппроксимацию усреднения экспоненциального ансамбля». Что из измеримого это подтверждает, а что остаётся интерпретацией?
> …

> **(b) Самый слабый эксперимент / чем бы фальсифицировала?** При каком датасете/режиме эффект НЕ проявится, и не станет ли это молчаливым условием?
> …

> **(c) Где сломалось / чего НЕ протестировали?** На «лёгкой» задаче без переобучения dropout ничего не даёт. Что это говорит о границах метода?
> …

## Зачем это

Широкая сеть на маленьких данных запоминает обучающую выборку, включая шум, и проваливается на тесте — классическое переобучение. Стандартный ответ до dropout — ансамбль: обучи много сетей, усредни предсказания. Дорого и по обучению, и по инференсу.

Фишка dropout: получить эффект ансамбля **одной** сетью. На каждом мини-батче случайно зануляем часть активаций — сеть каждый раз «прореженная», другая под-сеть с общими весами. За обучение проходит экспоненциальное семейство под-сетей, а на тесте работает полная сеть, приближающая их усреднение. Ни один нейрон не может полагаться на конкретного соседа (co-adaptation ломается), приходится учить устойчивые признаки.

**Почему такой датасет:** sklearn digits — настоящие 8×8 рукописные цифры, MNIST-подобные, но крохотные (1797 примеров) и потому CPU-дружелюбные. На чистой задаче она слишком лёгкая: ~91% test обеими руками, эффекта не видно. Чтобы воссоздать *режим переобучения* статьи на CPU за секунды, портим фиксированные 20% ОБУЧАЮЩИХ меток (тестовые чистые) — стандартный документированный способ форсировать запоминание. Незарегуляризованная сеть подгоняет шум (train acc → 1.0), test падает; dropout сопротивляется. Абсолютные числа MNIST мы НЕ копируем — воспроизводим направление и правдоподобную величину эффекта.

## Теория (Фейнман)

**Задача.** Есть слой активаций $x \in \mathbb{R}^{B \times d}$. Хотим на обучении случайно «выключать» единицы, чтобы сеть не переобучалась, но чтобы масштаб сигнала в среднем не поехал.

**Механизм (inverted dropout).** На обучении для каждой активации тянем независимую бернуллиеву маску: единица выживает с вероятностью $p$ (keep-prob), иначе зануляется. Выжившие масштабируем так, чтобы матожидание выхода равнялось входу:

$$\tilde x_i = \frac{m_i}{p}\, x_i,\qquad m_i \sim \mathrm{Bernoulli}(p),\qquad \mathbb{E}[m_i] = p.$$

Отсюда $\mathbb{E}[\tilde x_i] = \frac{x_i}{p}\,\mathbb{E}[m_i] = x_i$. Матожидание сохранено — поэтому на тесте **ничего масштабировать не надо**, слой работает как identity. Это и есть смысл слова *inverted*: цену масштабирования платим на обучении, а не на инференсе.

**Что метод измеряет.** Мы сравним две одинаковые сети (только $p$ отличается) по двум числам: train-test **gap** (насколько сеть переобучилась) и **test accuracy** (насколько обобщает). Эффект статьи = меньше gap и выше test-acc у руки с dropout.

*Где у меня дыра в понимании:* …

❓ **Inline Question 1.** Почему масштаб именно $1/p$, а не, скажем, $1/(1-p)$ или отсутствие масштаба вовсе? Что сломается на тесте, если масштаб убрать?

**Ваш ответ:** …

In [ ]:
# --- setup (готово, просто запусти) ---
# pip install numpy torch scikit-learn matplotlib
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cpu")
SEED = 0


def load_data(n_train: int = 400, n_test: int = 800, label_noise: float = 0.20):
    """sklearn digits в режиме переобучения: 20% ОБУЧАЮЩИХ меток рандомизированы
    (тест чистый), чтобы на CPU за секунды воспроизвести overfitting-режим статьи."""
    X, y = load_digits(return_X_y=True)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, train_size=n_train, test_size=n_test, stratify=y, random_state=SEED)
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr); X_te = scaler.transform(X_te)
    rng = np.random.default_rng(1000)
    y_tr = y_tr.copy()
    k = int(label_noise * len(y_tr))
    idx = rng.choice(len(y_tr), size=k, replace=False)
    y_tr[idx] = rng.integers(0, 10, size=k)
    to_t = lambda a, dt: torch.tensor(a, dtype=dt, device=DEVICE)
    return (to_t(X_tr, torch.float32), to_t(y_tr, torch.long),
            to_t(X_te, torch.float32), to_t(y_te, torch.long))


def majority_baseline(data) -> float:
    """Тривиальный пол: предсказываем самый частый класс обучающей выборки."""
    _, y_tr, _, y_te = data
    maj = torch.bincount(y_tr).argmax().item()
    return (y_te == maj).float().mean().item()


def accuracy(model, X, y) -> float:
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(1)
    return (pred == y).float().mean().item()


data = load_data()
print("train:", tuple(data[0].shape), "test:", tuple(data[2].shape),
      "| majority baseline test_acc =", round(majority_baseline(data), 3))

## Part 1 — оператор dropout (вклад статьи)

Вот несущая деталь. Реализуй `dropout_forward` в inverted-dropout конвенции: на обучении прорежаешь активации и компенсируешь масштаб так, чтобы матожидание выхода равнялось входу; на тесте — identity. Ключевой инвариант: `p_keep == 1.0` и `training == False` обязаны быть точным no-op (это «выключенная» рука ablation-а).

RNG передаётся снаружи (`generator`), чтобы маска была воспроизводимой и независимой от инициализации весов — тогда обе руки ablation стартуют с идентичного init.

In [ ]:
def dropout_forward(x: torch.Tensor, p_keep: float, training: bool,
                    generator: torch.Generator) -> torch.Tensor:
    """Inverted dropout — ядро механизма Srivastava et al. 2014.

    На обучении каждая активация независимо остаётся с вероятностью p_keep,
    остальные зануляются; выжившие компенсируются так, чтобы E[output] == input
    (поэтому на тесте пересчёт весов не нужен). На тесте — тождество.

    Args:
        x: [B, d] float — активации.
        p_keep: float в (0, 1] — вероятность сохранить единицу. 1.0 -> без dropout.
        training: bool — маску применяем только в режиме обучения.
        generator: torch.Generator — RNG для воспроизводимой маски (device x).
    Returns:
        [B, d] float, та же форма/dtype, что x; дифференцируемо по x.
    Note: p_keep == 1.0 и training == False обязаны быть ТОЧНЫМ no-op
          (это «выключенная» рука ablation-а).
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# (1) no-op инварианты: p_keep=1.0 и eval-режим не трогают активации
g = torch.Generator().manual_seed(0)
x = torch.randn(4, 7)
assert torch.equal(dropout_forward(x, 1.0, True, g), x), \
    "p_keep=1.0 должен быть ТОЧНЫМ no-op"
assert torch.equal(dropout_forward(x, 0.5, False, g), x), \
    "eval-режим (training=False) должен быть ТОЧНЫМ no-op"

# (2) сохранение матожидания inverted-dropout: E[out] ~= in
g = torch.Generator().manual_seed(1)
big = torch.ones(20000, 10)
out = dropout_forward(big, 0.5, True, g)
m = out.mean().item()
print(f"Expected E[out] \u2248 1.000 | у тебя {m:.3f}")
assert abs(m - 1.0) < 0.03, \
    f"inverted dropout должен сохранять среднее ~1.0, получено {m:.3f} (масштаб 1/p_keep?)"

# (3) реально зануляет ~половину при p_keep=0.5
frac_zero = (out == 0).float().mean().item()
print(f"Expected drop-fraction \u2248 0.500 (диапазон 0.45\u20130.55) | у тебя {frac_zero:.3f}")
assert 0.45 < frac_zero < 0.55, \
    f"при p_keep=0.5 должно зануляться ~50% активаций, получено {frac_zero:.3f}"
print("ok")

❓ **Inline Question 2.** RNG (`generator`) намеренно отделён от инициализации весов. Почему это важно именно для *честного* ablation on-vs-off — что было бы, дай мы маске и init один и тот же поток случайности?

**Ваш ответ:** …

In [ ]:
# --- MLP c твоим dropout между слоями (готово, просто запусти) ---
class MLP(nn.Module):
    """Двухслойный ReLU-MLP; dropout_forward между слоями (вход + два скрытых)."""
    def __init__(self, d_in, d_hidden, n_classes, p_keep_in, p_keep_hidden, seed):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_hidden)
        self.fc3 = nn.Linear(d_hidden, n_classes)
        self.p_keep_in = p_keep_in
        self.p_keep_hidden = p_keep_hidden
        # Отдельный генератор: поток маски воспроизводим и НЕзависим от init весов,
        # поэтому обе руки ablation делят идентичную инициализацию.
        self.gen = torch.Generator(device="cpu"); self.gen.manual_seed(seed + 1234)

    def forward(self, x):
        x = dropout_forward(x, self.p_keep_in, self.training, self.gen)
        x = F.relu(self.fc1(x))
        x = dropout_forward(x, self.p_keep_hidden, self.training, self.gen)
        x = F.relu(self.fc2(x))
        x = dropout_forward(x, self.p_keep_hidden, self.training, self.gen)
        return self.fc3(x)

print("MLP готов")

## Part 2 — training loop

Метод обучаемый — training loop пишешь ты. `train` гоняет `epochs` эпох Adam-ом: каждую эпоху перемешиваем индексы и идём мини-батчами по `batch`, на каждом батче — шаг оптимизации по cross-entropy. Перед проходом переведи модель в train-режим (иначе твой dropout не включится). После обучения верни train/test accuracy (функция `accuracy` дана в setup).

Перестановку бери из `np.random.default_rng(SEED)`, чтобы обе руки видели одинаковый порядок батчей.

In [ ]:
def train(model, X_tr, y_tr, X_te, y_te, epochs: int, lr: float, batch: int) -> dict:
    """Обучить модель мини-батчевым Adam-ом и вернуть финальные accuracy.

    Каждую эпоху: train-режим, свежая перестановка индексов из
    np.random.default_rng(SEED), проход мини-батчами размера `batch`,
    на каждом — шаг по F.cross_entropy.

    Args:
        model: MLP — обучаемая сеть.
        X_tr, y_tr: обучающие тензоры [N, d] float / [N] long.
        X_te, y_te: тестовые тензоры.
        epochs: int — число эпох.
        lr: float — learning rate для Adam.
        batch: int — размер мини-батча.
    Returns:
        dict с ключами 'train_acc' и 'test_acc' (float, через accuracy()).
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- обёртка сборки+обучения одной руки (готово) ---
def build_and_train(p_keep_in, p_keep_hidden, data, epochs) -> dict:
    X_tr, y_tr, X_te, y_te = data
    torch.manual_seed(SEED)  # идентичный init весов у обеих рук
    model = MLP(d_in=X_tr.shape[1], d_hidden=512, n_classes=10,
                p_keep_in=p_keep_in, p_keep_hidden=p_keep_hidden, seed=SEED).to(DEVICE)
    res = train(model, X_tr, y_tr, X_te, y_te, epochs=epochs, lr=1e-3, batch=32)
    res["gap"] = res["train_acc"] - res["test_acc"]
    return res

# смоук: одна короткая рука должна обучиться выше тривиального пола
_smoke = build_and_train(1.0, 1.0, data, epochs=10)
print("smoke (off, 10 эпох):", {k: round(v, 3) for k, v in _smoke.items()})
assert _smoke["test_acc"] > majority_baseline(data) + 0.30, \
    "даже без dropout сеть должна заметно бить majority-baseline — обучение сломано?"
print("ok")

❓ **Inline Question 3.** В `build_and_train` перед каждой рукой стоит `torch.manual_seed(SEED)`, а поток маски сидит на отдельном генераторе. Что именно это гарантирует про сравнение on-vs-off и почему без этого контраст было бы нечестно приписывать dropout-у?

**Ваш ответ:** …

## Part 3 — ablation: пощупать фишку

Теперь контролируемый контраст — сердце воспроизведения. Всё одинаково (архитектура, данные, оптимизатор, эпохи, seed, init), крутим единственную ручку `p_keep`:

- **off:** `p_keep=1.0` везде → твой `dropout_forward` становится точным no-op.
- **on:** канонические ставки статьи — 0.2 input drop (`p_keep_in=0.8`), 0.5 hidden drop (`p_keep_hidden=0.5`).

Ждём у руки on **меньший** train-test gap И **выше** test accuracy. Это и есть claim статьи, воспроизведённый как with-X vs without-X, а не «побили пол».

In [ ]:
EPOCHS = 100
off = build_and_train(1.0, 1.0, data, epochs=EPOCHS)   # dropout OFF (no-op)
on  = build_and_train(0.8, 0.5, data, epochs=EPOCHS)   # dropout ON (paper rates)
base = majority_baseline(data)

print("=" * 60)
print(f"{'arm':<14}{'train_acc':>11}{'test_acc':>11}{'gap':>9}")
print(f"{'no dropout':<14}{off['train_acc']:>11.3f}{off['test_acc']:>11.3f}{off['gap']:>9.3f}")
print(f"{'dropout':<14}{on['train_acc']:>11.3f}{on['test_acc']:>11.3f}{on['gap']:>9.3f}")
print("-" * 60)
print(f"gap reduction (off - on) = {off['gap'] - on['gap']:+.3f}")
print(f"test-acc gain (on - off) = {on['test_acc'] - off['test_acc']:+.3f}")
print(f"trivial majority baseline test_acc = {base:.3f}")
print("=" * 60)

In [ ]:
# Эффект статьи: dropout сужает train-test gap И поднимает test-acc.
# Пороги откалиброваны реальным прогоном reference (seed0: gap_red=+0.081, gain=+0.069;
# худший наблюдённый seed2: gap_red=0.036, gain=0.021 — пороги с положительным запасом).
print(f"Expected: gap(on) < gap(off) - 0.03 | on={on['gap']:.3f} off={off['gap']:.3f}")
assert on["gap"] < off["gap"] - 0.03, \
    f"dropout должен ЗАМЕТНО сузить gap: on={on['gap']:.3f} vs off={off['gap']:.3f}"

print(f"Expected: test_acc(on) > test_acc(off) + 0.02 | on={on['test_acc']:.3f} off={off['test_acc']:.3f}")
assert on["test_acc"] > off["test_acc"] + 0.02, \
    f"dropout должен ПОДНЯТЬ test-acc: on={on['test_acc']:.3f} off={off['test_acc']:.3f}"

# санити-пол: обе реальные сети далеко бьют тривиальный baseline
assert off["test_acc"] > base + 0.30 and on["test_acc"] > base + 0.30, \
    "обе сети обязаны крушить majority-baseline"
print("ok — эффект статьи воспроизведён: меньше gap, выше test-acc")

In [ ]:
# --- headline-фигура: gap и test-acc, off vs on (готово) ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
arms = ["no dropout", "dropout"]
ax[0].bar(arms, [off["gap"], on["gap"]], color=["#c44", "#48c"])
ax[0].set_title("train-test gap (меньше = лучше)"); ax[0].set_ylabel("gap")
ax[1].bar(arms, [off["test_acc"], on["test_acc"]], color=["#c44", "#48c"])
ax[1].axhline(base, ls="--", c="k", lw=1, label="majority baseline")
ax[1].set_title("test accuracy (больше = лучше)"); ax[1].set_ylabel("test_acc"); ax[1].legend()
plt.tight_layout(); plt.show()

**Сравни со статьёй.** Статья (MNIST, 2-слойная сеть): test error ~1.6% → ~1.3%, то есть меньше gap и ниже ошибка. Абсолютные числа у нас другие (иной датасет, крохотная сеть, инъекция шума), но **направление** обязано совпасть: gap у dropout-руки уже, test-acc выше. Совпало ли? Если у тебя gain по test-acc околонулевой — почти всегда `dropout_forward` не занулил ничего (проверь ветку no-op) или сеть не в train-режиме в `train`.

❓ **Inline Question 4.** На *чистой* задаче (без 20% шумных меток) dropout почти не двигает test-acc. В какой НОВОЙ ситуации ты потянешься за dropout, а в какой он окажется бесполезной или вредной добавкой?

**Ваш ответ:** …

## Рефлексия

**Что забрать в работу:** …

**В какой НОВОЙ ситуации dropout — правильный инструмент** (много параметров относительно данных, риск co-adaptation), а где ты предпочтёшь другую регуляризацию или больше данных: …

**Что осталось непонятным → следующий узел beam** (например: как dropout соотносится с BatchNorm, почему их часто не ставят вместе; или связь inverted-dropout с L2/weight decay как аппроксимацией байесова усреднения): …

## SPOILERS (открывай, только если застряла)

Начни сама — механизм короткий. Если совсем встал:

**`dropout_forward`.** Ранний выход-тождество, если `not training or p_keep >= 1.0`. Иначе тяни `torch.rand(x.shape, generator=generator, device=x.device)`, сравни с `p_keep` → булева маска выживших; верни `x * mask.to(x.dtype) / p_keep`. NaN/масштаб-дебаг: если среднее выхода не ~1.0 — забыл делить на `p_keep`; если no-op не точный — проверь оба условия ветки.

**`train`.** `opt = torch.optim.Adam(model.parameters(), lr=lr)`; `n = X_tr.shape[0]`; `rng = np.random.default_rng(SEED)`. Цикл по эпохам: `model.train()`, `perm = rng.permutation(n)`, внутренний цикл `for i in range(0, n, batch)`: `idx = perm[i:i+batch]`, `opt.zero_grad()`, `loss = F.cross_entropy(model(X_tr[idx]), y_tr[idx])`, `loss.backward()`, `opt.step()`. В конце вернуть `{'train_acc': accuracy(model, X_tr, y_tr), 'test_acc': accuracy(model, X_te, y_te)}`.